# Maximum detection efficiency for COM displacements

This notebook calculates the maximum detection efficiency associated with small center-of-mass displacements of a spherical nanoparticle.

## General imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
from pathlib import Path

## Repository imports and default configuration

In [3]:
import info_patterns.parameters as params
import info_patterns.constants as const
import info_patterns.generate_nanoparticle as gen
import info_patterns.light_matter_interaction_simulation as lmi
import info_patterns.information_patterns_simulation as ips
import info_patterns.measurement_tools as mt

## Nanoparticle geometry and material

In [4]:
geometry = gen.nanoparticle_geometry(**params.SPHERE_GEOMETRY_PARAMS) # Generate the discretized nanoparticle geometry 
material = gen.nanoparticle_material(params.DEFAULT_MATERIAL_NAME, **params.DEFAULT_MATERIAL_KWARGS) # Load the optical material model used by pyGDM

## Optical field and electromagnetic propagator

In [ ]:
# Far-field angular sampling parameters
farfield_params = params.DEFAULT_FARFIELD_PARAMS
Nteta = farfield_params["Nteta"]
Nphi = farfield_params["Nphi"]
r = farfield_params["r"]
field_index = farfield_params["field_index"]

efield = lmi.incident_field(**params.DEFAULT_GAUSSIAN_FIELD_PARAMS) # Incident optical field
dyads = lmi.field_propagation(**params.DEFAULT_DYADS_PARAMS) # Electromagnetic propagator/dyadic Green tensor configuration

# Simulator parameters
step_nm = params.SPHERE_GEOMETRY_PARAMS["step_nm"]
disp_nm = params.DEFAULT_COM_DISPLACEMENT_NM
wavelength_nm = params.DEFAULT_WAVELENGTH_NM

## COM information patterns

In [6]:
E_plus_x, E_minus_x, dE_x = lmi.com_scattered_farfield(geometry=geometry, step_nm=step_nm, material=material, efield=efield, dyads=dyads, axis_index=0, disp_nm=disp_nm, **farfield_params)
I_x_total, I_x_Ex, I_x_Ey, I_x_Ez = ips.info_patterns_from_scattered_field(dE=dE_x, delta_mu=disp_nm, wavelength_nm=wavelength_nm, Nteta=Nteta, Nphi=Nphi)
E_plus_y, E_minus_y, dE_y = lmi.com_scattered_farfield(geometry=geometry, step_nm=step_nm, material=material, efield=efield, dyads=dyads, axis_index=1, disp_nm=disp_nm, **farfield_params)
I_y_total, I_y_Ex, I_y_Ey, I_y_Ez = ips.info_patterns_from_scattered_field(dE=dE_y, delta_mu=disp_nm, wavelength_nm=wavelength_nm, Nteta=Nteta, Nphi=Nphi)
E_plus_z, E_minus_z, dE_z = lmi.com_scattered_farfield(geometry=geometry, step_nm=step_nm, material=material, efield=efield, dyads=dyads, axis_index=2, disp_nm=disp_nm, **farfield_params)
I_z_total, I_z_Ex, I_z_Ey, I_z_Ez = ips.info_patterns_from_scattered_field(dE=dE_z, delta_mu=disp_nm, wavelength_nm=wavelength_nm, Nteta=Nteta, Nphi=Nphi)

structure initialization - automatic mesh detection: hex
structure initialization - consistency check: 853/853 dipoles valid
timing for wl=1550.00nm - setup: EE 1605.6ms, inv.: 223.7ms, repropa.: 254.7ms (1 field configs), tot: 2084.3ms
structure initialization - automatic mesh detection: hex
structure initialization - consistency check: 853/853 dipoles valid
timing for wl=1550.00nm - setup: EE 50.8ms, inv.: 246.7ms, repropa.: 10.6ms (1 field configs), tot: 308.7ms
structure initialization - automatic mesh detection: hex
structure initialization - consistency check: 853/853 dipoles valid
timing for wl=1550.00nm - setup: EE 52.6ms, inv.: 264.8ms, repropa.: 7.2ms (1 field configs), tot: 324.7ms
structure initialization - automatic mesh detection: hex
structure initialization - consistency check: 853/853 dipoles valid
timing for wl=1550.00nm - setup: EE 70.3ms, inv.: 297.2ms, repropa.: 10.7ms (1 field configs), tot: 378.8ms
structure initialization - automatic mesh detection: hex
structur

## Maximum detection efficiency

We calculate the maximum detection efficiency for each COM displacement direction. The function `max_detection_efficiency` integrates each total information pattern over the selected collection region, defined here by `PLUS_Z_HEMISPHERE_THETA_MAX`. The printed output gives the detection efficiency as a percentage for displacements along `x`, `y`, and `z`.

In [7]:
hemisphere = params.PLUS_Z_HEMISPHERE_THETA_MAX

In [8]:
eta_x = mt.max_detection_efficiency(I_x_total, Nteta, Nphi, hemisphere)
eta_y = mt.max_detection_efficiency(I_y_total, Nteta, Nphi, hemisphere)
eta_z = mt.max_detection_efficiency(I_z_total, Nteta, Nphi, hemisphere)

print("Maximum detection efficiency")
print(f"x: {100 * eta_x:.2f} %")
print(f"y: {100 * eta_y:.2f} %")
print(f"z: {100 * eta_z:.2f} %")

Maximum detection efficiency
x: 49.55 %
y: 49.58 %
z: 89.97 %
